In [0]:
# Notebook parameters

params = {
    "proj_dir": "/Volumes/7_outgoing/pharos/imaging/pharos_20260619/",
    "output_path": "/Volumes/7_outgoing/pharos/imaging/pharos_20260619_metadata.parquet"
}

# create text widgets
for k in params.keys():
    dbutils.widgets.text(k, params[k], "")

# fetch values
for k in params.keys():
    params[k] = dbutils.widgets.get(k)
    print(k, ":", params[k])

In [0]:
from pyspark.sql.functions import lit, col, from_json, schema_of_json, regexp_replace
from pyspark.sql.types import StructType, StructField, StringType, TimestampType
import pandas as pd
import pydicom
from pydicom.errors import InvalidDicomError
from datetime import datetime
from typing import Iterator

In [0]:


# DataFrame containing DICOM file paths
files_df = (
    spark.read.format("binaryFile")
    .option("pathGlobFilter", "*.dcm")
    .option("recursiveFileLookup", "true")
    .option("includeBinaryFiles", "false")
    .load(params["proj_dir"])
    .select("path", "length")
)

from pyspark.sql.functions import expr
ROOT_DIR = params["proj_dir"]
files_df = files_df.withColumn("full_path", expr("substring(path, 6, length(path))"))
files_df = files_df.withColumn("path", regexp_replace("path", lit(ROOT_DIR), ""))

from pyspark.sql.functions import regexp_extract

files_df = files_df.withColumn(
    "subdir",
    regexp_extract("path", r"(/?\d{6}/\d{6}/)", 1)
)

# Output schema
schema = StructType([
    StructField("full_path", StringType(), True),
    StructField("path", StringType(), True),
    StructField("subdir", StringType(), True),
    StructField("accession_number", StringType(), True),
    StructField("study_datetime", TimestampType(), True),
    StructField("modality", StringType(), True),
    StructField("study_description", StringType(), True),
])

# mapInPandas function
def extract_dicom_tags(iterator: Iterator[pd.DataFrame]) -> Iterator[pd.DataFrame]:
    for pdf in iterator:
        results = []

        for _, row in pdf.iterrows():
            full_path = row["full_path"]
            path = row["path"]
            subdir = row["subdir"]

            try:
                ds = pydicom.dcmread(
                    full_path,
                    stop_before_pixels=True,
                    force=True
                )

                accession_number = str(ds.get("AccessionNumber", "")) or None
                modality = str(ds.get("Modality", "")) or None
                study_description = str(ds.get("StudyDescription", "")) or None

                # Build Timestamp from StudyDate + StudyTime
                study_datetime = None
                study_date = str(ds.get("StudyDate", "")).strip()
                study_time = str(ds.get("StudyTime", "")).split(".")[0].strip()

                if study_date:
                    try:
                        if study_time:
                            dt_str = study_date + study_time.ljust(6, "0")
                            study_datetime = datetime.strptime(
                                dt_str, "%Y%m%d%H%M%S"
                            )
                        else:
                            study_datetime = datetime.strptime(
                                study_date, "%Y%m%d"
                            )
                    except Exception:
                        study_datetime = None

                results.append({
                    "full_path": full_path,
                    "path": path,
                    "subdir": subdir,
                    "accession_number": accession_number,
                    "study_datetime": study_datetime,
                    "modality": modality,
                    "study_description": study_description,
                })

            except (InvalidDicomError, FileNotFoundError, Exception):
                results.append({
                    "full_path": full_path,
                    "path": path,
                    "subdir": subdir,
                    "accession_number": None,
                    "study_datetime": None,
                    "modality": None,
                    "study_description": None,
                })

        yield pd.DataFrame(results)

# Execute extraction
dicom_tags_df = files_df.mapInPandas(
    extract_dicom_tags,
    schema=schema
)

output_path = params["output_path"]
dicom_tags_df.write.mode("overwrite").parquet(output_path)